# Fiddler

>[Fiddler](https://www.fiddler.ai/) 是企业级生成式和预测式系统运维的先驱，提供了一个统一的平台，使数据科学、MLOps、风险、合规、分析以及其他业务线团队能够在企业规模上监控、解释、分析和改进 ML 部署。

## 1. 安装与设置

In [ ]:
#!pip install langchain langchain-community langchain-openai fiddler-client

## 2. Fiddler 连接详情

*在您能够添加有关您模型的 Fiddler 信息之前*

1. 您用于连接 Fiddler 的 URL
2. 您的组织 ID
3. 您的授权令牌

这些信息可以在您的 Fiddler 环境的*设置*页面中找到。

In [ ]:
URL = ""  # Your Fiddler instance URL, Make sure to include the full URL (including https://). For example: https://demo.fiddler.ai
ORG_NAME = ""
AUTH_TOKEN = ""  # Your Fiddler instance auth token

# Fiddler project and model names, used for model registration
PROJECT_NAME = ""
MODEL_NAME = ""  # Model name in Fiddler

## 3. 创建 fiddler 回调处理程序实例

```jsx
import { Fiddler } from '@alipay/fiddler-sdk';

const fiddler = new Fiddler({
  // ... fiddler configuration
});

In [ ]:
from langchain_community.callbacks.fiddler_callback import FiddlerCallbackHandler

fiddler_handler = FiddlerCallbackHandler(
    url=URL,
    org=ORG_NAME,
    project=PROJECT_NAME,
    model=MODEL_NAME,
    api_key=AUTH_TOKEN,
)

## 示例 1：基本链

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import OpenAI

# Note : Make sure openai API key is set in the environment variable OPENAI_API_KEY
llm = OpenAI(temperature=0, streaming=True, callbacks=[fiddler_handler])
output_parser = StrOutputParser()

chain = llm | output_parser

# Invoke the chain. Invocation will be logged to Fiddler, and metrics automatically generated
chain.invoke("How far is moon from earth?")

In [ ]:
# Few more invocations
chain.invoke("What is the temperature on Mars?")
chain.invoke("How much is 2 + 200000?")
chain.invoke("Which movie won the oscars this year?")
chain.invoke("Can you write me a poem about insomnia?")
chain.invoke("How are you doing today?")
chain.invoke("What is the meaning of life?")

## 示例 2：带提示词模板的链

This example demonstrates how to use prompt templates with LangChain.

本示例演示了如何将提示词模板与 LangChain 结合使用。

```python
from langchain.prompts import PromptTemplate
from langchain.llms import OpenAI
from langchain.chains import LLMChain

# Initialize the LLM
llm = OpenAI(temperature=0)

# Define the prompt template
template = """Question: {question}

Answer: """
prompt = PromptTemplate(
    input_variables=["question"],
    template=template,
)

# Create the LLMChain
chain = LLMChain(llm=llm, prompt=prompt)

# Run the chain
question = "What is the capital of France?"
response = chain.run(question)

print(f"Question: {question}")
print(f"Answer: {response}")
```

### Explanation

### 说明

1.  **`PromptTemplate`**: This class allows you to create reusable prompt templates. You can define variables that will be filled in when the chain is run.
    1.  **`PromptTemplate`**: 此类允许您创建可重用的提示词模板。您可以定义变量，这些变量将在链运行时被填充。

2.  **`LLM`**: This is the language model you want to use. In this case, we are using OpenAI's `text-davinci-003` model.
    2.  **`LLM`**: 这是您想使用的语言模型。在本例中，我们使用的是 OpenAI 的 `text-davinci-003` 模型。

3.  **`LLMChain`**: This is the core of the chain. It takes an LLM and a prompt template and combines them to create a chain that can be run.
    3.  **`LLMChain`**: 这是链的核心。它接收一个 LLM 和一个提示词模板，并将它们组合起来创建一个可以运行的链。

4.  **`chain.run(question)`**: This method runs the chain with the given input. The input `question` is passed to the `prompt` template, which then formats the prompt and sends it to the LLM. The LLM's response is then returned.
    4.  **`chain.run(question)`**: 此方法使用给定的输入运行链。输入 `question` 被传递给 `prompt` 模板，然后由模板格式化提示词并将其发送给 LLM。最后返回 LLM 的响应。

This example is a simple demonstration of how to use prompt templates with LangChain. You can create more complex chains by combining multiple prompt templates and other components.

这个示例是一个简单的演示，展示了如何将提示词模板与 LangChain 结合使用。您可以通过组合多个提示词模板和其他组件来创建更复杂的链。

In [ ]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    FewShotChatMessagePromptTemplate,
)

examples = [
    {"input": "2+2", "output": "4"},
    {"input": "2+3", "output": "5"},
]

example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a wondrous wizard of math."),
        few_shot_prompt,
        ("human", "{input}"),
    ]
)

# Note : Make sure openai API key is set in the environment variable OPENAI_API_KEY
llm = OpenAI(temperature=0, streaming=True, callbacks=[fiddler_handler])

chain = final_prompt | llm

# Invoke the chain. Invocation will be logged to Fiddler, and metrics automatically generated
chain.invoke({"input": "What's the square of a triangle?"})